<a href="https://colab.research.google.com/github/MariyamNP/task-Trading-Bot-on-Binance-Futures-Testnet/blob/main/Another_copy_of_Binance_Futures_Testnet_Bot_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Binance Futures Testnet Trading Bot



In [ ]:
import os
os.makedirs('bot', exist_ok=True)
print('Folders created!')

Folders created!


In [ ]:
%%writefile bot/__init__.py
# bot/__init__.py


Writing bot/__init__.py


In [ ]:
%%writefile bot/logging_config.py
"""
bot/logging_config.py
Centralised logging setup.
- INFO and above goes to trading_bot.log (rotating file)
- WARNING and above goes to console
"""

import logging
import logging.handlers
import os


def setup_logging(log_file='trading_bot.log', level=logging.DEBUG):
    log_dir = os.path.dirname(log_file)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)

    root_logger = logging.getLogger('trading_bot')
    root_logger.setLevel(level)

    if root_logger.handlers:
        return

    fmt = logging.Formatter(
        fmt='%(asctime)s [%(levelname)-8s] %(name)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
    )

    fh = logging.handlers.RotatingFileHandler(
        log_file, maxBytes=1_000_000, backupCount=5, encoding='utf-8'
    )
    fh.setLevel(level)
    fh.setFormatter(fmt)
    root_logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setLevel(logging.WARNING)
    ch.setFormatter(fmt)
    root_logger.addHandler(ch)

    root_logger.info('Logging initialised. File: %s', os.path.abspath(log_file))


Writing bot/logging_config.py


In [ ]:
%%writefile bot/validators.py
"""
bot/validators.py
Input validation helpers. Raise ValueError with a clear message on failure.
"""

from decimal import Decimal, InvalidOperation

VALID_SIDES = {'BUY', 'SELL'}
VALID_ORDER_TYPES = {'MARKET', 'LIMIT', 'STOP'}


def validate_symbol(symbol):
    symbol = symbol.strip().upper()
    if not symbol or not symbol.isalpha():
        raise ValueError(f"Invalid symbol '{symbol}'. Example: BTCUSDT")
    return symbol


def validate_side(side):
    side = side.strip().upper()
    if side not in VALID_SIDES:
        raise ValueError(f"Side must be one of {VALID_SIDES}, got '{side}'")
    return side


def validate_order_type(order_type):
    order_type = order_type.strip().upper()
    if order_type not in VALID_ORDER_TYPES:
        raise ValueError(f"Order type must be one of {VALID_ORDER_TYPES}, got '{order_type}'")
    return order_type


def validate_quantity(quantity):
    try:
        qty = Decimal(str(quantity))
        if qty <= 0:
            raise ValueError('Quantity must be greater than 0')
        return str(qty)
    except InvalidOperation:
        raise ValueError(f"Invalid quantity '{quantity}'. Must be a positive number.")


def validate_price(price):
    try:
        p = Decimal(str(price))
        if p <= 0:
            raise ValueError('Price must be greater than 0')
        return str(p)
    except InvalidOperation:
        raise ValueError(f"Invalid price '{price}'. Must be a positive number.")


def validate_order_params(symbol, side, order_type, quantity, price=None, stop_price=None):
    validated = {
        'symbol':   validate_symbol(symbol),
        'side':     validate_side(side),
        'type':     validate_order_type(order_type),
        'quantity': validate_quantity(quantity),
    }
    if validated['type'] == 'LIMIT':
        if price is None:
            raise ValueError('Price is required for LIMIT orders.')
        validated['price'] = validate_price(price)
    if validated['type'] == 'STOP':
        if price is None:
            raise ValueError('Limit price is required for STOP orders.')
        if stop_price is None:
            raise ValueError('stop_price is required for STOP orders.')
        validated['price']     = validate_price(price)
        validated['stopPrice'] = validate_price(stop_price)
    return validated

Writing bot/validators.py


In [ ]:
%%writefile bot/client.py
"""
bot/client.py
Binance Futures Testnet REST API client.
Handles HMAC-SHA256 signing and HTTP communication.
"""

import hashlib
import hmac
import time
import logging
import requests
from urllib.parse import urlencode

logger = logging.getLogger('trading_bot.client')

TESTNET_BASE_URL = 'https://testnet.binancefuture.com'


class BinanceClient:
    def __init__(self, api_key, api_secret, base_url=TESTNET_BASE_URL):
        self.api_key = api_key
        self.api_secret = api_secret
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.session.headers.update({
            'X-MBX-APIKEY': self.api_key,
            'Content-Type': 'application/x-www-form-urlencoded',
        })
        logger.info('BinanceClient initialised. Base URL: %s', self.base_url)

    def _timestamp(self):
        return int(time.time() * 1000)

    def _sign(self, params):
        params['timestamp'] = self._timestamp()
        query_string = urlencode(params)
        signature = hmac.new(
            self.api_secret.encode('utf-8'),
            query_string.encode('utf-8'),
            hashlib.sha256,
        ).hexdigest()
        params['signature'] = signature
        return params

    def _request(self, method, endpoint, params=None, signed=False):
        params = params or {}
        if signed:
            params = self._sign(params)
        url = self.base_url + endpoint
        logger.debug('-> %s %s  params=%s', method, url,
                     {k: v for k, v in params.items() if k != 'signature'})
        try:
            if method.upper() == 'GET':
                response = self.session.get(url, params=params, timeout=10)
            elif method.upper() == 'POST':
                response = self.session.post(url, data=params, timeout=10)
            elif method.upper() == 'DELETE':
                response = self.session.delete(url, params=params, timeout=10)
            else:
                raise ValueError(f'Unsupported HTTP method: {method}')
            logger.debug('<- %s %s', response.status_code, response.text[:500])
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError as exc:
            try:
                error_body = exc.response.json()
            except Exception:
                error_body = exc.response.text
            logger.error('HTTP error %s: %s', exc.response.status_code, error_body)
            raise
        except requests.exceptions.ConnectionError as exc:
            logger.error('Network connection error: %s', exc)
            raise
        except requests.exceptions.Timeout as exc:
            logger.error('Request timed out: %s', exc)
            raise

    def get_server_time(self):
        return self._request('GET', '/fapi/v1/time')

    def get_account_info(self):
        return self._request('GET', '/fapi/v2/account', signed=True)

    def get_exchange_info(self):
        return self._request('GET', '/fapi/v1/exchangeInfo')

    def place_order(self, params):
        logger.info('Placing order: %s', params)
        return self._request('POST', '/fapi/v1/order', params=params, signed=True)

    def get_open_orders(self, symbol=None):
        params = {}
        if symbol:
            params['symbol'] = symbol
        return self._request('GET', '/fapi/v1/openOrders', params=params, signed=True)

    def cancel_order(self, symbol, order_id):
        params = {'symbol': symbol, 'orderId': order_id}
        return self._request('DELETE', '/fapi/v1/order', params=params, signed=True)

Writing bot/client.py


In [ ]:
%%writefile bot/orders.py
import logging
from bot.validators import validate_order_params

logger = logging.getLogger('trading_bot.orders')

def _build_order_params(symbol, side, order_type, quantity, price=None, stop_price=None):
    validated = validate_order_params(symbol, side, order_type, quantity, price, stop_price)
    params = {
        'symbol':   validated['symbol'],
        'side':     validated['side'],
        'type':     validated['type'],
        'quantity': validated['quantity'],
    }
    if order_type == 'LIMIT':
        params['price']       = validated['price']
        params['timeInForce'] = 'GTC'
    if order_type == 'STOP':
        params['price']       = validated['price']
        params['stopPrice']   = validated['stopPrice']
        params['timeInForce'] = 'GTC'
    return params

def print_order_summary(params):
    print('\n' + '=' * 50)
    print('  ORDER REQUEST SUMMARY')
    print('=' * 50)
    for label, key in [('Symbol','symbol'),('Side','side'),('Type','type'),
                        ('Quantity','quantity'),('Price','price'),('Stop Price','stopPrice')]:
        if key in params:
            print(f'  {label:<12}: {params[key]}')
    print('=' * 50 + '\n')

def print_order_response(response):
    print('\n' + '=' * 50)
    print('  ORDER RESPONSE')
    print('=' * 50)
    print(f"  Order ID     : {response.get('orderId')}")
    print(f"  Symbol       : {response.get('symbol')}")
    print(f"  Side         : {response.get('side')}")
    print(f"  Type         : {response.get('type')}")
    print(f"  Status       : {response.get('status')}")
    print(f"  Orig Qty     : {response.get('origQty')}")
    print(f"  Executed Qty : {response.get('executedQty')}")
    print(f"  Avg Price    : {response.get('avgPrice') or response.get('price','N/A')}")
    print('=' * 50 + '\n')

def place_order(client, symbol, side, order_type, quantity, price=None, stop_price=None):
    params = _build_order_params(symbol, side, order_type, quantity, price, stop_price)
    print_order_summary(params)
    logger.info('Submitting %s %s: symbol=%s qty=%s price=%s stop_price=%s',
                order_type, side, symbol, quantity, price or 'N/A', stop_price or 'N/A')
    try:
        response = client.place_order(params)
        logger.info('Order placed. orderId=%s status=%s', response.get('orderId'), response.get('status'))
        print_order_response(response)
        print('Order placed successfully!')
        return response
    except Exception as exc:
        logger.error('Order failed: %s', exc)
        print(f'Order failed: {exc}')
        raise


Writing bot/orders.py


In [ ]:
import importlib
import bot.validators, bot.orders
importlib.reload(bot.validators)
importlib.reload(bot.orders)
from bot.orders import place_order
print('Reloaded!')

Reloaded!


In [ ]:
!pip install requests -q
print('requests installed!')

requests installed!


In [ ]:
import os

# Paste your testnet credentials here
os.environ['BINANCE_API_KEY']    = 'joX99896xohVOb7gPHrQJpyyp3usqqEtad5u2iwnYxSdGXLnAT4G7qYcCLMKrah7'
os.environ['BINANCE_API_SECRET'] = 'v3ksw0qoGejDdd66otmsgoZ1vhzcICNSWy869ChU39UOm1ZodqFgPP19BQe95PIC'

print('Credentials set.')

Credentials set.


In [ ]:
import sys
sys.path.insert(0, '/content')

from bot.client import BinanceClient
from bot.logging_config import setup_logging

setup_logging('trading_bot.log')

client = BinanceClient(
    api_key    = os.environ['BINANCE_API_KEY'],
    api_secret = os.environ['BINANCE_API_SECRET'],
)

server_time = client.get_server_time()
print(f"Server time (ms): {server_time['serverTime']}")
print('Connected to Binance Futures Testnet!')

INFO:trading_bot:Logging initialised. File: /content/trading_bot.log
INFO:trading_bot.client:BinanceClient initialised. Base URL: https://testnet.binancefuture.com
DEBUG:trading_bot.client:-> GET https://testnet.binancefuture.com/fapi/v1/time  params={}
DEBUG:trading_bot.client:<- 200 {"serverTime":1777648825985}


Server time (ms): 1777648825985
Connected to Binance Futures Testnet!


In [ ]:
info = client.get_account_info()
print(f"Total Wallet Balance : {info.get('totalWalletBalance')} USDT")
print(f"Available Balance    : {info.get('availableBalance')} USDT")
print(f"Total Unrealised PnL : {info.get('totalUnrealizedProfit')} USDT")

DEBUG:trading_bot.client:-> GET https://testnet.binancefuture.com/fapi/v2/account  params={'timestamp': 1777648830933}
DEBUG:trading_bot.client:<- 200 {"feeTier":0,"canTrade":true,"canDeposit":true,"canWithdraw":true,"feeBurn":true,"tradeGroupId":-1,"updateTime":0,"multiAssetsMargin":false,"totalInitialMargin":"11.76540147","totalMaintMargin":"0.94123211","totalWalletBalance":"4999.90710832","totalUnrealizedProfit":"3.07882922","totalMarginBalance":"5002.98593754","totalPositionInitialMargin":"11.76540147","totalOpenOrderInitialMargin":"0.00000000","totalCrossWalletBalance":"4999.90710832","totalCrossUnPnl":"3.07882922","availableBalance":"499


Total Wallet Balance : 4999.90710832 USDT
Available Balance    : 4991.11190685 USDT
Total Unrealised PnL : 3.07882922 USDT


In [ ]:
from bot.orders import place_order

# Edit these values
SYMBOL   = 'BTCUSDT'
SIDE     = 'BUY'    # BUY or SELL
QUANTITY = '0.001'  # in BTC

response = place_order(
    client     = client,
    symbol     = SYMBOL,
    side       = SIDE,
    order_type = 'MARKET',
    quantity   = QUANTITY,
)

INFO:trading_bot.orders:Submitting MARKET BUY: symbol=BTCUSDT qty=0.001 price=N/A stop_price=N/A
INFO:trading_bot.client:Placing order: {'symbol': 'BTCUSDT', 'side': 'BUY', 'type': 'MARKET', 'quantity': '0.001'}
DEBUG:trading_bot.client:-> POST https://testnet.binancefuture.com/fapi/v1/order  params={'symbol': 'BTCUSDT', 'side': 'BUY', 'type': 'MARKET', 'quantity': '0.001', 'timestamp': 1777648835781}
DEBUG:trading_bot.client:<- 200 {"orderId":13096852307,"symbol":"BTCUSDT","status":"NEW","clientOrderId":"liZCeNYnSItA422WwK8xXH","price":"0.00","avgPrice":"0.00","origQty":"0.0010","executedQty":"0.0000","cumQty":"0.0000","cumQuote":"0.000000","timeInForce":"GTC","type":"MARKET","reduceOnly":false,"closePosition":false,"side":"BUY","positionSide":"BOTH","stopPrice":"0.00","workingType":"CONTRACT_PRICE","priceProtect":false,"origType":"MARKET","priceMatch":"NONE","selfTradePreventionMode":"EXPIRE_MAKER","goodTillDate":0,"updat
INFO:trading_bot.orders:Order placed. orderId=13096852307 stat


  ORDER REQUEST SUMMARY
  Symbol      : BTCUSDT
  Side        : BUY
  Type        : MARKET
  Quantity    : 0.001


  ORDER RESPONSE
  Order ID     : 13096852307
  Symbol       : BTCUSDT
  Side         : BUY
  Type         : MARKET
  Status       : NEW
  Orig Qty     : 0.0010
  Executed Qty : 0.0000
  Avg Price    : 0.00

Order placed successfully!


In [ ]:
# Edit these values
SYMBOL   = 'BTCUSDT'
SIDE     = 'SELL'
QUANTITY = '0.001'
PRICE    = '120000'  # your desired limit price in USDT

response = place_order(
    client     = client,
    symbol     = SYMBOL,
    side       = SIDE,
    order_type = 'LIMIT',
    quantity   = QUANTITY,
    price      = PRICE,
)

INFO:trading_bot.orders:Submitting LIMIT SELL: symbol=BTCUSDT qty=0.001 price=120000 stop_price=N/A
INFO:trading_bot.client:Placing order: {'symbol': 'BTCUSDT', 'side': 'SELL', 'type': 'LIMIT', 'quantity': '0.001', 'price': '120000', 'timeInForce': 'GTC'}
DEBUG:trading_bot.client:-> POST https://testnet.binancefuture.com/fapi/v1/order  params={'symbol': 'BTCUSDT', 'side': 'SELL', 'type': 'LIMIT', 'quantity': '0.001', 'price': '120000', 'timeInForce': 'GTC', 'timestamp': 1777648870211}
DEBUG:trading_bot.client:<- 200 {"orderId":13096853565,"symbol":"BTCUSDT","status":"NEW","clientOrderId":"zm9DahyBZKUOCq475ewene","price":"120000.00","avgPrice":"0.00","origQty":"0.0010","executedQty":"0.0000","cumQty":"0.0000","cumQuote":"0.000000","timeInForce":"GTC","type":"LIMIT","reduceOnly":false,"closePosition":false,"side":"SELL","positionSide":"BOTH","stopPrice":"0.00","workingType":"CONTRACT_PRICE","priceProtect":false,"origType":"LIMIT","priceMatch":"NONE","selfTradePreventionMode":"EXPIRE_MAKE


  ORDER REQUEST SUMMARY
  Symbol      : BTCUSDT
  Side        : SELL
  Type        : LIMIT
  Quantity    : 0.001
  Price       : 120000


  ORDER RESPONSE
  Order ID     : 13096853565
  Symbol       : BTCUSDT
  Side         : SELL
  Type         : LIMIT
  Status       : NEW
  Orig Qty     : 0.0010
  Executed Qty : 0.0000
  Avg Price    : 0.00

Order placed successfully!


In [ ]:
import json
orders = client.get_open_orders(symbol='BTCUSDT')
if orders:
    for o in orders:
        print(json.dumps(o, indent=2))
else:
    print('No open orders.')

DEBUG:trading_bot.client:-> GET https://testnet.binancefuture.com/fapi/v1/openOrders  params={'symbol': 'BTCUSDT', 'timestamp': 1777648916267}
DEBUG:trading_bot.client:<- 200 [{"orderId":13096398328,"symbol":"BTCUSDT","status":"NEW","clientOrderId":"Qx6IcjDmz7mtls06hLdq6p","price":"120000","avgPrice":"0","origQty":"0.0010","executedQty":"0","cumQuote":"0.000000","timeInForce":"GTC","type":"LIMIT","reduceOnly":false,"closePosition":false,"side":"SELL","positionSide":"BOTH","stopPrice":"0","workingType":"CONTRACT_PRICE","priceProtect":false,"origType":"LIMIT","priceMatch":"NONE","selfTradePreventionMode":"EXPIRE_MAKER","goodTillDate":0,"time":1777636960240,"updateTime"


{
  "orderId": 13096398328,
  "symbol": "BTCUSDT",
  "status": "NEW",
  "clientOrderId": "Qx6IcjDmz7mtls06hLdq6p",
  "price": "120000",
  "avgPrice": "0",
  "origQty": "0.0010",
  "executedQty": "0",
  "cumQuote": "0.000000",
  "timeInForce": "GTC",
  "type": "LIMIT",
  "reduceOnly": false,
  "closePosition": false,
  "side": "SELL",
  "positionSide": "BOTH",
  "stopPrice": "0",
  "workingType": "CONTRACT_PRICE",
  "priceProtect": false,
  "origType": "LIMIT",
  "priceMatch": "NONE",
  "selfTradePreventionMode": "EXPIRE_MAKER",
  "goodTillDate": 0,
  "time": 1777636960240,
  "updateTime": 1777636960240
}
{
  "orderId": 13096426255,
  "symbol": "BTCUSDT",
  "status": "NEW",
  "clientOrderId": "XSBwXu8EtGVygoIPK1KlIl",
  "price": "120000",
  "avgPrice": "0",
  "origQty": "0.0010",
  "executedQty": "0",
  "cumQuote": "0.000000",
  "timeInForce": "GTC",
  "type": "LIMIT",
  "reduceOnly": false,
  "closePosition": false,
  "side": "SELL",
  "positionSide": "BOTH",
  "stopPrice": "0",
  "wor

In [ ]:
from google.colab import files

with open('trading_bot.log') as f:
    lines = f.readlines()
print(''.join(lines[-30:]))

files.download('trading_bot.log')

2026-05-01 15:20:25 [INFO    ] trading_bot - Logging initialised. File: /content/trading_bot.log
2026-05-01 15:20:25 [INFO    ] trading_bot.client - BinanceClient initialised. Base URL: https://testnet.binancefuture.com
2026-05-01 15:20:25 [DEBUG   ] trading_bot.client - -> GET https://testnet.binancefuture.com/fapi/v1/time  params={}
2026-05-01 15:20:26 [DEBUG   ] trading_bot.client - <- 200 {"serverTime":1777648825985}
2026-05-01 15:20:30 [DEBUG   ] trading_bot.client - -> GET https://testnet.binancefuture.com/fapi/v2/account  params={'timestamp': 1777648830933}
2026-05-01 15:20:31 [DEBUG   ] trading_bot.client - <- 200 {"feeTier":0,"canTrade":true,"canDeposit":true,"canWithdraw":true,"feeBurn":true,"tradeGroupId":-1,"updateTime":0,"multiAssetsMargin":false,"totalInitialMargin":"11.76540147","totalMaintMargin":"0.94123211","totalWalletBalance":"4999.90710832","totalUnrealizedProfit":"3.07882922","totalMarginBalance":"5002.98593754","totalPositionInitialMargin":"11.76540147","totalOpe

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>